# Quilt-VQA Evaluation (Pillar 1b) — Benchmarks 1 and 3

Implements Sub-pillar 1b (Section 2.1 / Table 2, Figure 3): once a judge is validated
against pathologist consensus on the PathOPEN/filtered-PathVQA subset (see
`judge_pathologist_agreement.ipynb`), apply it to score Quilt-VQA on the **same
benchmarks and criteria** PathOPEN and filtered-PathVQA were scored on.

Quilt-VQA (`wisdomik/Quilt_VQA` on HuggingFace, already cached locally) has 985 rows,
split by `answer_type` into 724 `OPEN` and 261 `CLOSED`. The two splits are scored on
**different benchmarks**, mirroring how the other two datasets are handled:

| split | n | benchmark | criteria |
|---|---|---|---|
| `OPEN` | 724 | 1 | Knowledge Interpretation/Deduction, Visual Grounding |
| `CLOSED` | 257 | 3 | Visual Grounding/Reasoning |

The paper's stated Sub-pillar 1b scope is "Quilt-VQA's open-ended pairs" and Table 3
marks Quilt-VQA as CE ✗ — but Quilt-VQA *does* have close-ended questions; their answers
simply store the yes/no verdict with an explanation attached ("Yes, hyperchromasia and
enlargement are visible in the image."). Splitting the verdict off recovers a genuine
close-ended item, giving a like-for-like Benchmark 3 comparison against PathOPEN CE and
filtered-PathVQA CE. See the "Load Quilt-VQA" section for the supporting counts.

Quilt-VQA has no wrong answers, no MCQ, and no case/image-ID scheme like PathOPEN - each
row is a standalone (image, question, answer) triple, so no external image resolver is
needed here (images are embedded directly in the HF dataset).

**Prerequisite**: `judge_pathologist_agreement.ipynb` should have already established
judge-vs-pathologist weighted kappa for Benchmarks 1 and 3 before treating these scores
as meaningful - this notebook does not re-run that validation, it assumes it has been
done and reports the judges' Quilt-VQA scores alongside a reminder of where to find it.

## Checkpointing

One judge call per scored row (981 = 724 + 257 per judge; 4 CLOSED rows have no
recoverable verdict and are excluded). Every call is checkpointed immediately to
`checkpoints/{model_key}_quiltvqa.jsonl`, keyed by the row's index in the **unfiltered**
dataset, so the key is stable no matter how the strata are filtered later. Re-running the
scoring cell after an interruption skips whatever's already checkpointed.

## Outputs

- `judge_output/evaluator_{model_key}/quiltvqa_eval_data.csv` - per-row scores, both
  strata. Same layout as the PathOPEN and PathVQA runners: model as directory, dataset
  as filename.
- `agreement_output/pathopen_vs_quiltvqa_eval_mannwhitney.csv` - the PathOPEN-vs-Quilt-VQA
  distribution comparison, one row per (judge × stratum × criterion), next to
  `pathopen_vs_pathvqa_mannwhitney.csv` from `judge_pathologist_agreement.ipynb`.

In [1]:
import os
import sys

sys.path.insert(0, os.getcwd())  # so gpu_allocation/judge_models resolve when the CWD is this dir
from gpu_allocation import cuda_visible_devices_for, describe_allocation, max_memory_for

# Which judge(s) this kernel will load. Declared HERE, before torch touches CUDA,
# because CUDA_VISIBLE_DEVICES has no effect once CUDA is initialized - if a torch CUDA
# op has already run in this kernel, restart it.
#
# Both judges run unquantized at bf16 (~64 GB Qwen / ~76 GB InternVL), so each is
# sharded over 3 of the 47.4 GiB A6000s. They are placed in different NUMA islands
# (Qwen 0-2, InternVL 4-6) so they can run concurrently without sharing a PCIe switch.
MODELS_TO_RUN = ["qwenvl", "internvl"]

os.environ["CUDA_VISIBLE_DEVICES"] = cuda_visible_devices_for(*MODELS_TO_RUN)
print(describe_allocation())

PER_CARD_MEMORY = 42GiB
  qwenvl    -> physical GPUs [0, 1, 2]  [NUMA island A (0-3)]
  internvl  -> physical GPUs [4, 5, 6]  [NUMA island B (4-7)]
  CUDA_VISIBLE_DEVICES currently = '0,1,2,4,5,6'


In [2]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from checkpoint import JudgeCheckpoint
from judge_models import JudgeModel
from parallel_judges import run_judges_in_parallel
from prompts.benchmarks import (
    BENCHMARK_1,
    BENCHMARK_3,
    build_benchmark_1_prompt,
    build_benchmark_3_prompt,
)

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Same layout as the PathOPEN and PathVQA runners: per-row judge scores go to
# judge_output/evaluator_{model_key}/{dataset}_eval_data.csv (model as directory, dataset
# as filename), and cross-dataset statistics go to agreement_output/ alongside
# pathopen_vs_pathvqa_mannwhitney.csv from judge_pathologist_agreement.ipynb.
JUDGE_OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
AGREEMENT_OUTPUT_DIR = os.path.join(os.getcwd(), "agreement_output")
CHECKPOINT_DIR = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(AGREEMENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def judge_output_path(model_key: str, filename: str) -> str:
    """judge_output/evaluator_{model_key}/{filename}, creating the per-judge dir."""
    directory = os.path.join(JUDGE_OUTPUT_DIR, f"evaluator_{model_key}")
    os.makedirs(directory, exist_ok=True)
    return os.path.join(directory, filename)


## Load Quilt-VQA — OPEN on Benchmark 1, CLOSED on Benchmark 3

Quilt-VQA's 985 rows split by `answer_type` into 724 `OPEN` and 261 `CLOSED`. The two
splits are scored on **different benchmarks**, matching how PathOPEN and filtered-PathVQA
are treated:

| split | n | benchmark | criteria |
|---|---|---|---|
| `OPEN` | 724 | 1 | Knowledge Interpretation/Deduction, Visual Grounding |
| `CLOSED` | 261 | 3 | Visual Grounding/Reasoning |

The `CLOSED` answers are *not* stored as bare yes/no — they are a yes/no verdict followed
by an explanation, e.g. *"Yes, hyperchromasia and enlargement are visible in the image."*
But the verdict is recoverable, and reliably so:

- **257/261 (98.5%)** begin with `yes` (190) or `no` (67); only **4** do not.
- **0/724** `OPEN` answers begin with yes/no, so `answer_type` partitions cleanly and no
  open-ended row can leak into the close-ended stratum.
- Stripping the prefix never empties an answer — every row keeps its explanation.

So the yes/no prefix is split off to give a genuine close-ended answer, which makes this
an apple-to-apple comparison against PathOPEN CE (yes/no "with no further explanation
required", paper §3.2) and filtered-PathVQA CE (selected by `answer.lower() in
("yes","no")`). Benchmark 3 grades whether the *question* requires image+text to reach
that yes/no, which is exactly what these items now support.

The 4 rows without a recoverable verdict are excluded from the CE stratum and reported,
not silently dropped.

In [4]:
import collections
import re

quilt_vqa = load_dataset("wisdomik/Quilt_VQA")["train"]
quilt_vqa_all = quilt_vqa

# Leading yes/no verdict, plus any trailing punctuation/whitespace before the explanation.
_YESNO_PREFIX = re.compile(r"^\s*(yes|no)\b[\s,.:;!-]*", re.IGNORECASE)


def split_yes_no(answer: str):
    """('yes'|'no'|None, remaining explanation).

    Quilt-VQA stores CLOSED answers as a verdict followed by its justification
    ("Yes, hyperchromasia and enlargement are visible in the image."). Splitting the
    verdict off recovers a true close-ended answer, comparable to PathOPEN CE and
    filtered-PathVQA CE, both of which are bare yes/no.
    """
    text = str(answer).strip()
    m = _YESNO_PREFIX.match(text)
    if not m:
        return None, text
    return m.group(1).lower(), text[m.end():].strip()


quilt_vqa_open = quilt_vqa_all.filter(lambda r: r["answer_type"] == "OPEN")
_closed_all = quilt_vqa_all.filter(lambda r: r["answer_type"] == "CLOSED")
# Only rows with a recoverable verdict can be scored as close-ended.
quilt_vqa_closed = _closed_all.filter(lambda r: split_yes_no(r["answer"])[0] is not None)

print(f"total {len(quilt_vqa_all)} | OPEN {len(quilt_vqa_open)} | CLOSED {len(_closed_all)}")
print(f"CLOSED with recoverable yes/no verdict: {len(quilt_vqa_closed)}/{len(_closed_all)}"
      f" ({len(quilt_vqa_closed)/len(_closed_all):.1%})")
print("  verdicts:", dict(collections.Counter(split_yes_no(a)[0] for a in quilt_vqa_closed["answer"])))

# Guard the partition: an OPEN answer starting with yes/no would contaminate the CE
# stratum. Verified 0/724 at time of writing - assert so a dataset revision cannot
# silently break the split.
_open_yesno = sum(1 for a in quilt_vqa_open["answer"] if split_yes_no(a)[0] is not None)
print(f"OPEN answers starting with yes/no (expect 0): {_open_yesno}")
assert _open_yesno == 0, "OPEN rows now look close-ended; revisit the OPEN/CLOSED split"

_dropped = [a for a in _closed_all["answer"] if split_yes_no(a)[0] is None]
if _dropped:
    print(f"\nexcluded from the CE stratum ({len(_dropped)} rows, no yes/no verdict):")
    for a in _dropped:
        print("   ", str(a)[:100])

Filter: 100%|██████████| 261/261 [00:02<00:00, 120.91 examples/s]

total 985 | OPEN 724 | CLOSED 261
CLOSED with recoverable yes/no verdict: 257/261 (98.5%)
  verdicts: {'yes': 190, 'no': 67}
OPEN answers starting with yes/no (expect 0): 0

excluded from the CE stratum (4 rows, no yes/no verdict):
    The image appears to show approximately an equal distribution of tumor and non-tumor cells.
    There is no pleomorphism observed in the image because these are translocation sarcomas, which usual
    The image is showing a momentous calcification.
    The image does not provide a clear answer to whether it shows muscle or thin ropes of collagen again


## Load judge models

In [ ]:
# MODELS_TO_RUN is set in the first cell (it has to be, to pick GPUs before CUDA init).
# max_memory_for(...) returns LOGICAL device ids - CUDA_VISIBLE_DEVICES renumbers cards,
# so with "4,5,6" visible torch sees 0,1,2.
judges = {
    key: JudgeModel(key, max_memory=max_memory_for(key, MODELS_TO_RUN))
    for key in MODELS_TO_RUN
}

# Verify each judge loaded as intended BEFORE committing to a multi-hour run.
# device_map="auto" silently offloads to CPU/disk when a model does not fit, which turns
# a long run into a multi-day one - catch it here, not hours in.
import collections

import torch

for key, judge in judges.items():
    devs = collections.Counter(str(p.device) for p in judge.model.parameters())
    offloaded = [d for d in devs if d in ("cpu", "meta", "disk")]
    print(f"{key}: devices={dict(devs)}")
    print(f"    system_role={judge.use_system_role}  images_in_template={judge.images_in_template}"
          f"  thinking={'<think>' in judge.system_prompt}")
    print(f"    OFFLOAD -> {offloaded if offloaded else 'none (good)'}")
    if offloaded:
        raise RuntimeError(f"{key} offloaded to {offloaded} - lower max_memory or add a card")

judges

## Score Quilt-VQA: Benchmark 1 on the 724 OPEN rows, Benchmark 3 on the 257 CLOSED rows

In [ ]:
def build_quiltvqa_task(row):
    """(benchmark, prompt, criteria, yes_no, explanation) for one row.

    OPEN  -> Benchmark 1 on the answer as written.
    CLOSED -> Benchmark 3 on the yes/no verdict split out of the answer, so the item
              matches PathOPEN CE / filtered-PathVQA CE, which are bare yes/no.
    Returns None for a CLOSED row with no recoverable verdict (4 rows) - excluded from
    the CE stratum rather than scored on a benchmark that does not fit.
    """
    if row["answer_type"] == "OPEN":
        return (1, build_benchmark_1_prompt(row["question"], row["answer"]),
                list(BENCHMARK_1["criteria"].keys()), None, None)

    yes_no, explanation = split_yes_no(row["answer"])
    if yes_no is None:
        return None
    return (3, build_benchmark_3_prompt(row["question"], yes_no),
            list(BENCHMARK_3["criteria"].keys()), yes_no, explanation)


def run_quiltvqa_scoring(judge: JudgeModel, model_key: str) -> JudgeCheckpoint:
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_quiltvqa.jsonl"))
    n_scored = n_skipped = n_failed = n_excluded = 0

    # item_id is the row's index in the UNFILTERED dataset, so it stays stable no matter
    # how the OPEN/CLOSED strata are filtered downstream.
    for row_idx, row in enumerate(tqdm(quilt_vqa_all, desc=f"{model_key} Quilt-VQA")):
        item_id = row_idx
        if checkpoint.is_done(item_id):
            n_skipped += 1
            continue
        task = build_quiltvqa_task(row)
        if task is None:
            n_excluded += 1
            continue
        benchmark, prompt, criteria, yes_no, explanation = task
        try:
            scores, raw = judge.score(row["image"], prompt, criteria)
        except Exception as e:
            n_failed += 1
            print(f"[checkpoint] FAILED item_id={item_id!r}: {e!r} - will retry next run")
            continue
        checkpoint.append({
            "item_id": item_id,
            "answer_type": row["answer_type"],   # OPEN / CLOSED
            "benchmark": benchmark,              # 1 for OPEN, 3 for CLOSED
            "question": row["question"],
            "answer": row["answer"],             # original, unmodified
            "yes_no": yes_no,                    # CLOSED only: the verdict actually scored
            "explanation": explanation,          # CLOSED only: the stripped remainder
            "scores": scores,
            "raw_response": raw,
        })
        n_scored += 1
        print(f"[{n_scored}/{len(quilt_vqa_all)}][checkpoint] {model_key} quiltvqa: "
              f"item_id={item_id} {row['answer_type']}/B{benchmark} - {scores}")

    print(f"{model_key} Quilt-VQA: scored {n_scored} new, {n_skipped} already done, "
          f"{n_failed} failed, {n_excluded} excluded (no yes/no verdict)")
    return checkpoint


# Judges run concurrently - separate GPUs, separate checkpoint files. See parallel_judges.py.
quiltvqa_checkpoints = run_judges_in_parallel(run_quiltvqa_scoring, judges, MODELS_TO_RUN)

## Post-process: assemble the checkpoint into a CSV

Pure re-read of the checkpoint **file** (via a fresh `JudgeCheckpoint(path)`, not
the in-memory `quiltvqa_checkpoints` object from the scoring cell above) - safe to
re-run any time, independent of the scoring cell above, even in a fresh kernel.

In [ ]:
def assemble_quiltvqa_csv(model_key: str) -> pd.DataFrame:
    """Reads checkpoints/{model_key}_quiltvqa.jsonl directly from disk - does NOT
    depend on the `quiltvqa_checkpoints` dict from the scoring cell, so this is
    safe to run standalone in a fresh kernel.

    One table holds both strata; the Benchmark-1 columns are populated for OPEN rows
    and the Benchmark-3 column for CLOSED rows, so a row's benchmark is never ambiguous.
    """
    checkpoint = JudgeCheckpoint(os.path.join(CHECKPOINT_DIR, f"{model_key}_quiltvqa.jsonl"))
    records = []
    for record in sorted(checkpoint.load_all(), key=lambda r: r["item_id"]):
        scores = record["scores"]
        records.append({
            "row_uid": record["item_id"],
            "answer_type": record.get("answer_type"),
            "benchmark": record.get("benchmark"),
            "question": record["question"],
            "answer": record["answer"],
            "yes_no": record.get("yes_no"),
            # Benchmark 1 (OPEN)
            "Knowledge_Interpretation_Deduction": scores.get("Knowledge Interpretation/Deduction"),
            "Visual_Grounding": scores.get("Visual Grounding"),
            # Benchmark 3 (CLOSED)
            "Visual_Grounding_Reasoning": scores.get("Visual Grounding/Reasoning"),
        })
    return pd.DataFrame.from_records(records)


# Named "eval_data", not "benchmark1": this table holds BOTH the Benchmark 1 scores
# (OPEN rows) and the Benchmark 3 scores (CLOSED rows). Written to
# judge_output/evaluator_{model_key}/quiltvqa_eval_data.csv - the same layout the PathOPEN
# and PathVQA runners use, with the model as a directory and the dataset as the filename.
for model_key in MODELS_TO_RUN:
    out_df = assemble_quiltvqa_csv(model_key)
    out_path = judge_output_path(model_key, "quiltvqa_eval_data.csv")
    out_df.to_csv(out_path, index=False)
    print(model_key, "->", out_path, out_df.shape)
    if len(out_df):
        print("   rows per stratum:", out_df.groupby(["answer_type", "benchmark"]).size().to_dict())
        b1 = out_df[out_df["benchmark"] == 1]
        b3 = out_df[out_df["benchmark"] == 3]
        if len(b1):
            print("   B1 means:", b1[["Knowledge_Interpretation_Deduction", "Visual_Grounding"]].mean().round(3).to_dict())
        if len(b3):
            print("   B3 mean :", round(b3["Visual_Grounding_Reasoning"].mean(), 3),
                  "| verdicts:", b3["yes_no"].value_counts().to_dict())


## Figure 3 Panel B equivalent: PathOPEN vs. Quilt-VQA, judge-rated

Reuses the same Mann-Whitney U / rank-biserial approach as
`judge_pathologist_agreement.ipynb`'s PathOPEN-vs-filtered-PathVQA comparison,
substituting Quilt-VQA as the comparator and using each judge's **own** PathOPEN scores
(from the PathOPEN runner's output) as the PathOPEN side - so the comparison is
judge-vs-judge-on-two-datasets, matching the paper's description ("Forest plot,
judge-rated Benchmark 1 scores, PathOPEN vs. Quilt-VQA").

Each stratum is compared against the PathOPEN column scored on the **same benchmark**:
`OPEN` vs PathOPEN's OE correct answers (Benchmark 1), `CLOSED` vs PathOPEN's CE answers
(Benchmark 3). This extends the paper's stated Panel B, which covers Benchmark 1 only.

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu

# JUDGE_OUTPUT_DIR / judge_output_path come from the config cell above.


def rank_biserial_from_u(u_stat: float, n1: int, n2: int) -> float:
    return 1 - (2 * u_stat) / (n1 * n2)


def compare_distributions(a: pd.Series, b: pd.Series) -> dict:
    """Mann-Whitney U over the FULL ordinal scale {-1, 0, 1, 2}.

    -1 is kept, not filtered. It is a defined rubric level ("Unable to comprehend the
    question and/or image, or unable to make the evaluation"), not a missing value: the
    human pathologists assigned it to PathVQA items whose question or answer did not make
    sense, so it is a property of the data, and it is exactly the kind of defect this
    dataset-vs-dataset comparison exists to surface. Dropping it would delete the finding
    and bias each mean upward in proportion to how defective that dataset actually is.

    This includes the -1s that `JudgeModel.score` synthesizes when parsing fails after
    both retries. Those are not clerical noise: all 10 such Quilt-VQA cases have
    46k-61k-character raw responses, i.e. the judge reasoned to the 12,288-token retry
    ceiling and still never committed to a final JSON - operationally the same "unable to
    make the evaluation" the rubric describes, usually on a genuinely confusing item.

    Only genuinely non-numeric cells are dropped (`errors="coerce"` -> NaN)."""
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 2 or len(b) < 2:
        return {"n_pathopen": len(a), "n_quilt": len(b), "mean_pathopen": np.nan,
                "mean_quilt": np.nan, "n_neg1_pathopen": 0, "n_neg1_quilt": 0,
                "u_stat": np.nan, "p_value": np.nan, "rank_biserial": np.nan}
    u_stat, p_value = mannwhitneyu(a, b, alternative="two-sided")
    return {"n_pathopen": len(a), "n_quilt": len(b),
            "mean_pathopen": round(a.mean(), 3), "mean_quilt": round(b.mean(), 3),
            # Surfaced so a mean pulled down by unscorable items is never mistaken for a
            # mean pulled down by poor-but-scorable ones - different claims about a dataset.
            "n_neg1_pathopen": int((a == -1).sum()), "n_neg1_quilt": int((b == -1).sum()),
            "u_stat": u_stat, "p_value": p_value,
            "rank_biserial": round(rank_biserial_from_u(u_stat, len(a), len(b)), 3)}


# Each Quilt-VQA stratum is compared against the PathOPEN column scored on the SAME
# benchmark: OPEN vs PathOPEN's OE correct answers (B1), CLOSED vs PathOPEN's CE
# answers (B3). Comparing across benchmarks would not be meaningful.
COMPARISONS = [
    # (stratum, benchmark, quilt column, PathOPEN column, criterion label)
    ("OPEN", 1, "Knowledge_Interpretation_Deduction",
     'Evaluation OE_Correct_Answer_1\n(Benchmark 1)', "Knowledge Interpretation/Deduction"),
    ("OPEN", 1, "Visual_Grounding",
     "OE_Correct_Answer_1_VisGround", "Visual Grounding"),
    ("CLOSED", 3, "Visual_Grounding_Reasoning",
     'Evaluation CE_Correct_Answer\n(Benchmark 3)', "Visual Grounding/Reasoning"),
]

forest_rows = []
for model_key in MODELS_TO_RUN:
    quilt_path = judge_output_path(model_key, "quiltvqa_eval_data.csv")
    pathopen_path = judge_output_path(model_key, "pathopen_eval_data.csv")
    missing = [p for p in (quilt_path, pathopen_path) if not os.path.exists(p)]
    if missing:
        print(f"Skipping {model_key}: not found -> {missing}")
        continue
    quilt_df = pd.read_csv(quilt_path)
    pathopen_df = pd.read_csv(pathopen_path)

    for stratum, benchmark, quilt_col, pathopen_col, criterion in COMPARISONS:
        subset = quilt_df[quilt_df["answer_type"] == stratum]
        result = compare_distributions(pathopen_df[pathopen_col], subset[quilt_col])
        result.update({"judge": model_key, "stratum": stratum, "benchmark": benchmark,
                       "criterion": criterion})
        forest_rows.append(result)

forest_results = pd.DataFrame(forest_rows)
if len(forest_results):
    forest_results = forest_results[["judge", "stratum", "benchmark", "criterion",
                                     "n_pathopen", "n_quilt", "mean_pathopen", "mean_quilt",
                                     "n_neg1_pathopen", "n_neg1_quilt",
                                     "u_stat", "p_value", "rank_biserial"]]
    print(forest_results.to_string(index=False))
    print("\nSign convention: rb = 1 - 2U/(n_a*n_b), so a NEGATIVE rank_biserial means "
          "PathOPEN ranks HIGHER than Quilt-VQA. Read it against the two mean columns.")
    print("-1 ('unable to comprehend/evaluate') is INCLUDED as the bottom of the ordinal "
          "scale; n_neg1_* shows how much of each mean it accounts for.")
forest_results


In [ ]:
# Cross-dataset statistics live in agreement_output/, next to
# pathopen_vs_pathvqa_mannwhitney.csv from judge_pathologist_agreement.ipynb - same kind
# of artifact, same place. Covers both benchmarks (B1 on OPEN, B3 on CLOSED), so the
# filename says "eval" rather than naming a single benchmark.
forest_results.to_csv(
    os.path.join(AGREEMENT_OUTPUT_DIR, "pathopen_vs_quiltvqa_eval_mannwhitney.csv"), index=False)


## Reminder: this notebook does not establish judge credibility on its own

The judge-vs-pathologist weighted kappa (the actual validation step Sub-pillar 1b
requires before trusting these Quilt-VQA scores) lives in
`judge_pathologist_agreement.ipynb`'s `all_agreement` table - filtered to
`benchmark == 1` for the OPEN stratum and `benchmark == 3` for the CLOSED stratum.
Report both together, as the paper's Figure 3 does (Panel A: judge-vs-pathologist kappa
on the validation subset; Panel B: judge-rated scores, PathOPEN vs. Quilt-VQA).

Note that Benchmark 3's kappa is the weaker of the two: PathOPEN CE human scores are
96.9% twos, so kappa collapses toward zero there for prevalence reasons and exact
agreement / Gwet's AC1 should be reported alongside it.

## Raw checkpoint files

`checkpoints/{model_key}_quiltvqa.jsonl` retains every judge call (question, original
answer, the split-out yes/no verdict and explanation for CLOSED rows, which benchmark was
applied, scores, and the full raw model response) and is never overwritten.